# Phase 2 — Hyperparameter Search

Implements EXPERIMENT_PLAN §2.1–§2.4. Reads `runs/_phase1_calibration.json` for
the budget anchor `N` and eval cadence `X`, runs the HP sweep across the
baseline and fine-tune conditions, and writes `runs/_phase2_results.json` with
the winning HP config per condition + the pretrained checkpoint paths feeding
Phase 3.

Every cell is resumable: an existing `eval_log.jsonl` (or a saved pretrained
checkpoint) causes that run to be skipped, so the notebook can be killed and
restarted freely.

## 0. Setup

In [1]:
import json, math, pathlib, time
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

from experiment.config import ExperimentConfig, FRICTION_NORMAL, FRICTION_SLIPPERY
from experiment.train import train
from experiment.run_io import read_jsonl

# ---- Phase 1 anchors ---------------------------------------------------
CALIB_PATH = pathlib.Path("runs/_phase1_calibration.json")
if not CALIB_PATH.exists():
    raise FileNotFoundError(
        f"{CALIB_PATH} not found. Run phase1_calibration.ipynb first."
    )
calib = json.loads(CALIB_PATH.read_text())
N: int = int(calib["N"])     # baseline / pretrain / fine-tune budget unit
X: int = int(calib["X"])     # eval cadence (env steps between checkpoints)

print(f"N = {N:,} env steps   (2N = {2*N:,} for matched-compute baselines)")
print(f"X = {X:,} env steps   (~{N // X} eval points per N-step run)")

# ---- top-level knobs ---------------------------------------------------
HP_SEEDS      = (1,)              # 3 validation seeds per plan §5, 1 for sanity check
DEVICE        = "cpu"
SAVE_DIR      = "runs"

# Subdirs under SAVE_DIR for each section's outputs
SEARCH_SUBDIR    = "phase2_search"             # §2.2 baseline search
PRETRAIN_SUBDIR  = "phase2_pretrain"           # §2.3 winning-HP pretrain
FT_SEARCH_SUBDIR = "phase2_finetune_search"    # §2.4 fine-tune search

CHECKPOINTS_DIR = pathlib.Path("checkpoints")
CHECKPOINTS_DIR.mkdir(exist_ok=True)
PHASE2_RESULTS  = pathlib.Path(SAVE_DIR) / "_phase2_results.json"

# Search-eval cadence: same X as calibration. The env+greedy policy are
# deterministic (no env RNG, see car_env.py), so eval_n_episodes=1 gives
# bit-identical estimates to n_episodes=10 — we save 10x eval compute.
SEARCH_EVAL_EVERY = X
SEARCH_EVAL_N_EP  = 1

N = 250,000 env steps   (2N = 500,000 for matched-compute baselines)
X = 10,000 env steps   (~25 eval points per N-step run)


## 1. Define the HP grids (§2.1)

Two grids per plan §5:
- **Baseline / pretrain grid** (6 configs): `LR × ε-decay-end-step` — used for B-normal, B-slippery, B-DR.
- **Fine-tune grid** (9 configs; plan says "~6"): `LR × ε-start`, with ε-decay fixed at 30% of fine-tune budget per plan §5.

In [2]:
def _hp_id(prefix, **kv):
    """Stable, sortable, filename-safe HP config ID."""
    parts = [prefix]
    for k, v in kv.items():
        parts.append(f"{k}{v:g}" if isinstance(v, float) else f"{k}{v}")
    return "-".join(parts)

def baseline_grid(budget):
    """6 configs: 3 LR × 2 ε-decay-end-step. `budget` is the run's
    `max_env_steps` (N for B-normal, 2N for B-slippery/B-DR)."""
    configs = []
    for lr in (1e-3, 3e-4, 1e-4):
        for dec_pct in (30, 60):
            configs.append({
                "id":                  _hp_id("b", lr=lr, dec=f"{dec_pct}pct"),
                "lr":                  lr,
                "epsilon_start":       1.0,
                "epsilon_end":         0.05,
                "epsilon_decay_steps": int(dec_pct / 100 * budget),
            })
    return configs[:1] # only 1 for sanity check

def finetune_grid(budget):
    """9 configs: 3 LR × 3 ε-start. ε-decay fixed at 30% of budget.
    LR ≤ pretrain LR range per plan §5."""
    configs = []
    dec_steps = int(0.30 * budget)
    for lr in (3e-4, 1e-4, 3e-5):
        for eps_start in (0.5, 0.2):
            configs.append({
                "id":                  _hp_id("f", lr=lr, eps=eps_start),
                "lr":                  lr,
                "epsilon_start":       eps_start,
                "epsilon_end":         0.05,
                "epsilon_decay_steps": dec_steps,
            })
    return configs[:1] # only 1 for sanity check

print(f"Baseline grid (N={N:,}-budget):")
for c in baseline_grid(N): print("  ", c)
print(f"\nFine-tune grid (N={N:,}-budget):")
for c in finetune_grid(N): print("  ", c)

Baseline grid (N=250,000-budget):
   {'id': 'b-lr0.001-dec30pct', 'lr': 0.001, 'epsilon_start': 1.0, 'epsilon_end': 0.05, 'epsilon_decay_steps': 75000}

Fine-tune grid (N=250,000-budget):
   {'id': 'f-lr0.0003-eps0.5', 'lr': 0.0003, 'epsilon_start': 0.5, 'epsilon_end': 0.05, 'epsilon_decay_steps': 75000}


## 2. Search the baseline conditions (§2.2)

Three conditions, each searched independently:

| Condition | Friction | Budget |
|---|---|---|
| **B-normal**   | fixed 0.1                  | N  |
| **B-slippery** | fixed 0.95                 | 2N (matched-compute) |
| **B-DR**       | Uniform(0.1, 0.95) per ep  | 2N (matched-compute) |

Winner per condition is picked in the next cell by AUC of the on-target eval
curve (plan §2.2 asymmetry: B-normal scores on normal eval; the slippery-target
conditions score on slippery eval).

In [3]:
BASELINE_CONDITIONS = [
    # (name,        friction_kwargs,                                  budget, target_friction_for_AUC)
    ("B-normal",    dict(friction_mode="fixed", friction=FRICTION_NORMAL),                         N,     FRICTION_NORMAL),
    ("B-slippery",  dict(friction_mode="fixed", friction=FRICTION_SLIPPERY),                       2 * N, FRICTION_SLIPPERY),
    ("B-DR",        dict(friction_mode="dr",
                         friction_dr_low=FRICTION_NORMAL,
                         friction_dr_high=FRICTION_SLIPPERY),                                      2 * N, FRICTION_SLIPPERY),
]

def _build_baseline_cfg(condition_name, friction_kwargs, budget, hp, seed):
    return ExperimentConfig(
        run_name             = f"{SEARCH_SUBDIR}/{condition_name}/{hp['id']}",
        seed                 = seed,
        save_dir             = SAVE_DIR,
        device               = DEVICE,
        max_env_steps        = budget,
        eval_every_env_steps = SEARCH_EVAL_EVERY,
        eval_n_episodes      = SEARCH_EVAL_N_EP,
        eval_frictions       = (FRICTION_NORMAL, FRICTION_SLIPPERY),
        lr                   = hp["lr"],
        epsilon_start        = hp["epsilon_start"],
        epsilon_end          = hp["epsilon_end"],
        epsilon_decay_steps  = hp["epsilon_decay_steps"],
        **friction_kwargs,
    )

def _eval_log_path(cfg):
    return pathlib.Path(cfg.save_dir) / cfg.run_name / f"seed_{cfg.seed}" / "eval_log.jsonl"

total = sum(len(baseline_grid(b)) for _, _, b, _ in BASELINE_CONDITIONS) * len(HP_SEEDS)
print(f"Baseline search: {total} runs.\n"
      f"Existing runs (eval_log.jsonl present) are skipped.\n")

t0 = time.time()
n_done = n_skipped = 0
for condition_name, friction_kwargs, budget, _ in BASELINE_CONDITIONS:
    for hp in baseline_grid(budget):
        for seed in HP_SEEDS:
            cfg = _build_baseline_cfg(condition_name, friction_kwargs, budget, hp, seed)
            tag = f"[{condition_name:<10s} {hp['id']:<20s} seed={seed}]"
            if _eval_log_path(cfg).exists():
                n_skipped += 1
                continue
            print(f"  run {tag}  budget={cfg.max_env_steps:,}")
            train(cfg, verbose=False)
            n_done += 1

print(f"\nBaseline search done: {n_done} run, {n_skipped} skipped."
      f"  Elapsed: {(time.time()-t0)/60:.1f} min")

Baseline search: 3 runs.
Existing runs (eval_log.jsonl present) are skipped.

  run [B-normal   b-lr0.001-dec30pct   seed=1]  budget=250,000


[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.
[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.


  run [B-slippery b-lr0.001-dec30pct   seed=1]  budget=500,000


[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.
[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.


  run [B-DR       b-lr0.001-dec30pct   seed=1]  budget=500,000


[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.
[CarRacingEnv] 66 START tiles found; using first at row=975, col=170.



Baseline search done: 3 run, 0 skipped.  Elapsed: 45.6 min


### 2b. Pick the winning HP config per baseline condition

Metric: trapezoidal AUC of the eval curve at the target friction,
averaged across the 3 validation seeds. Tie-break by lower std.

B-normal scores on **normal** eval (its own target); B-slippery and B-DR both
target slippery generalization so they score on **slippery** eval.

In [ ]:
def _curve_at_friction(log, target_f):
    """Mean-across-maps eval return per checkpoint at one friction."""
    steps, vals = [], []
    for rec in log:
        cells = [c for c in rec["cells"] if math.isclose(c["friction"], target_f)]
        if not cells: continue
        steps.append(rec["env_steps"])
        vals.append(float(np.mean([c["return_mean"] for c in cells])))
    return np.array(steps), np.array(vals)

def _auc(steps, vals):
    """Trapezoidal AUC, normalised by the swept x-range."""
    if len(steps) < 2: return float("nan")
    # NumPy 2.0 renamed `np.trapz` -> `np.trapezoid`. Same semantics.
    return float(np.trapezoid(vals, steps) / (steps[-1] - steps[0]))

baseline_winners = {}
print(f"{'condition':<11} {'winning hp':<22} {'AUC target':<10} {'mean AUC':>10} {'std':>8}")
print("-" * 72)
for condition_name, friction_kwargs, budget, target_friction in BASELINE_CONDITIONS:
    grid = baseline_grid(budget)
    scored = {}
    for hp in grid:
        per_seed = []
        for seed in HP_SEEDS:
            cfg = _build_baseline_cfg(condition_name, friction_kwargs, budget, hp, seed)
            log_path = _eval_log_path(cfg)
            if not log_path.exists():
                per_seed.append(float("nan")); continue
            steps, vals = _curve_at_friction(read_jsonl(log_path), target_friction)
            per_seed.append(_auc(steps, vals))
        arr = np.array(per_seed, dtype=float)
        scored[hp["id"]] = {
            "mean_auc":     float(np.nanmean(arr)),
            "std_auc":      float(np.nanstd(arr)),
            "per_seed_auc": [float(v) for v in arr],
        }
    best_id, best = max(scored.items(),
                        key=lambda kv: (kv[1]["mean_auc"], -kv[1]["std_auc"]))
    baseline_winners[condition_name] = {
        "hp_id":            best_id,
        "hp_config":        next(h for h in grid if h["id"] == best_id),
        "target_friction":  target_friction,
        "mean_auc":         best["mean_auc"],
        "std_auc":          best["std_auc"],
        "per_seed_auc":     best["per_seed_auc"],
        "search_budget":    budget,
        "all_configs_auc":  scored,
    }
    print(f"{condition_name:<11} {best_id:<22} f={target_friction:<8.2f} "
          f"{best['mean_auc']:>10.1f} {best['std_auc']:>8.1f}")

## 3. Pretrain on normal with the winning B-normal HPs (§2.3)

3 seeds at full budget `N`. Final checkpoints are copied to
`checkpoints/pretrained_normal_seed_{seed}.pt` so the four fine-tune
conditions below can each load the seed-matched checkpoint.

In [ ]:
PRETRAIN_SEEDS = HP_SEEDS  # 3 per §2.3; Phase 3 will re-pretrain with 5 fresh

def _pretrained_ckpt_path(seed):
    return CHECKPOINTS_DIR / f"pretrained_normal_seed_{seed}.pt"

best_normal = baseline_winners["B-normal"]["hp_config"]
print(f"Pretrain HPs (winning B-normal): {best_normal}\n")

for seed in PRETRAIN_SEEDS:
    out = _pretrained_ckpt_path(seed)
    if out.exists():
        print(f"  skip seed={seed} (have {out})")
        continue
    cfg = ExperimentConfig(
        run_name             = PRETRAIN_SUBDIR,
        seed                 = seed,
        save_dir             = SAVE_DIR,
        device               = DEVICE,
        friction_mode        = "fixed",
        friction             = FRICTION_NORMAL,
        max_env_steps        = N,
        eval_every_env_steps = X,
        eval_n_episodes      = SEARCH_EVAL_N_EP,
        eval_frictions       = (FRICTION_NORMAL, FRICTION_SLIPPERY),
        lr                   = best_normal["lr"],
        epsilon_start        = best_normal["epsilon_start"],
        epsilon_end          = best_normal["epsilon_end"],
        epsilon_decay_steps  = best_normal["epsilon_decay_steps"],
    )
    print(f"  pretrain seed={seed}  budget={N:,}")
    train(cfg, verbose=False)
    src = pathlib.Path(SAVE_DIR) / cfg.run_name / f"seed_{seed}" / "final_checkpoint.pt"
    out.write_bytes(src.read_bytes())
    print(f"    → wrote {out}")

## 4. Search the fine-tune conditions (§2.4)

Four conditions in a 2×2 ablation (direct vs curriculum × full vs freeze-1st):

| | Full fine-tune | Freeze first layer |
|---|---|---|
| **Direct** (fixed f=0.95)          | E1 | E2 |
| **Curriculum** (0.1 → 0.95 linear) | E3 | E4 |

Each fine-tune run loads `checkpoints/pretrained_normal_seed_{seed}.pt`
(seed-matched) and trains for `N` env steps.

In [ ]:
FT_CONDITIONS = [
    # (name, friction_kwargs, freeze_first)
    ("E1", dict(friction_mode="fixed",      friction=FRICTION_SLIPPERY),                                                       False),
    ("E2", dict(friction_mode="fixed",      friction=FRICTION_SLIPPERY),                                                       True),
    ("E3", dict(friction_mode="curriculum", friction_curr_start=FRICTION_NORMAL, friction_curr_end=FRICTION_SLIPPERY),         False),
    ("E4", dict(friction_mode="curriculum", friction_curr_start=FRICTION_NORMAL, friction_curr_end=FRICTION_SLIPPERY),         True),
]
FT_BUDGET = N

def _build_ft_cfg(condition_name, friction_kwargs, hp, seed):
    return ExperimentConfig(
        run_name             = f"{FT_SEARCH_SUBDIR}/{condition_name}/{hp['id']}",
        seed                 = seed,
        save_dir             = SAVE_DIR,
        device               = DEVICE,
        max_env_steps        = FT_BUDGET,
        eval_every_env_steps = SEARCH_EVAL_EVERY,
        eval_n_episodes      = SEARCH_EVAL_N_EP,
        eval_frictions       = (FRICTION_NORMAL, FRICTION_SLIPPERY),
        lr                   = hp["lr"],
        epsilon_start        = hp["epsilon_start"],
        epsilon_end          = hp["epsilon_end"],
        epsilon_decay_steps  = hp["epsilon_decay_steps"],
        **friction_kwargs,
    )

# Pretrained checkpoints must exist for every HP_SEED we're about to fine-tune.
missing = [s for s in HP_SEEDS if not _pretrained_ckpt_path(s).exists()]
if missing:
    raise RuntimeError(
        f"Pretrained checkpoints missing for seeds {missing}. Re-run section 3."
    )

total = len(FT_CONDITIONS) * len(finetune_grid(FT_BUDGET)) * len(HP_SEEDS)
print(f"Fine-tune search: {total} runs.\n")

t0 = time.time()
n_done = n_skipped = 0
for condition_name, friction_kwargs, freeze_first in FT_CONDITIONS:
    for hp in finetune_grid(FT_BUDGET):
        for seed in HP_SEEDS:
            cfg = _build_ft_cfg(condition_name, friction_kwargs, hp, seed)
            tag = f"[{condition_name} {hp['id']:<20s} seed={seed} freeze={int(freeze_first)}]"
            if _eval_log_path(cfg).exists():
                n_skipped += 1
                continue
            print(f"  run {tag}")
            train(
                cfg,
                pretrained_ckpt = str(_pretrained_ckpt_path(seed)),
                freeze_first    = freeze_first,
                verbose         = False,
            )
            n_done += 1

print(f"\nFine-tune search done: {n_done} run, {n_skipped} skipped."
      f"  Elapsed: {(time.time()-t0)/60:.1f} min")

### 4b. Pick the winning HP config per fine-tune condition

All four E-conditions target slippery, so all score on slippery AUC.

In [ ]:
ft_winners = {}
print(f"{'cond':<5} {'winning hp':<22} {'mean AUC (slippery)':>22} {'std':>8} {'freeze':>8}")
print("-" * 72)
for condition_name, friction_kwargs, freeze_first in FT_CONDITIONS:
    grid = finetune_grid(FT_BUDGET)
    scored = {}
    for hp in grid:
        per_seed = []
        for seed in HP_SEEDS:
            cfg = _build_ft_cfg(condition_name, friction_kwargs, hp, seed)
            log_path = _eval_log_path(cfg)
            if not log_path.exists():
                per_seed.append(float("nan")); continue
            steps, vals = _curve_at_friction(read_jsonl(log_path), FRICTION_SLIPPERY)
            per_seed.append(_auc(steps, vals))
        arr = np.array(per_seed, dtype=float)
        scored[hp["id"]] = {
            "mean_auc":     float(np.nanmean(arr)),
            "std_auc":      float(np.nanstd(arr)),
            "per_seed_auc": [float(v) for v in arr],
        }
    best_id, best = max(scored.items(),
                        key=lambda kv: (kv[1]["mean_auc"], -kv[1]["std_auc"]))
    ft_winners[condition_name] = {
        "hp_id":           best_id,
        "hp_config":       next(h for h in grid if h["id"] == best_id),
        "freeze_first":    bool(freeze_first),
        "friction_kwargs": friction_kwargs,
        "mean_auc":        best["mean_auc"],
        "std_auc":         best["std_auc"],
        "per_seed_auc":    best["per_seed_auc"],
        "search_budget":   FT_BUDGET,
        "all_configs_auc": scored,
    }
    print(f"{condition_name:<5} {best_id:<22} {best['mean_auc']:>22.1f} {best['std_auc']:>8.1f} {str(freeze_first):>8}")

## 5. Visualise the search

One bar per HP config per condition, mean AUC ± std across the 3 validation
seeds. Winners highlighted. Useful for spotting whether the pick was clear-cut
or near-tied (and thus seed-dependent).

In [ ]:
def _plot_search(ax, scored, winner_id, title):
    ids = list(scored.keys())
    means = [scored[i]["mean_auc"] for i in ids]
    stds  = [scored[i]["std_auc"]  for i in ids]
    colors = ["tab:orange" if i == winner_id else "tab:gray" for i in ids]
    xs = np.arange(len(ids))
    ax.bar(xs, means, yerr=stds, color=colors, capsize=3)
    ax.set_xticks(xs); ax.set_xticklabels(ids, rotation=70, ha="right", fontsize=8)
    ax.set_ylabel("mean AUC")
    ax.set_title(title)
    ax.grid(alpha=0.3, axis="y")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, _, _, _) in zip(axes, BASELINE_CONDITIONS):
    w = baseline_winners[name]
    _plot_search(ax, w["all_configs_auc"], w["hp_id"], f"{name}  (AUC @ f={w['target_friction']})")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, _, _) in zip(axes, FT_CONDITIONS):
    w = ft_winners[name]
    _plot_search(ax, w["all_configs_auc"], w["hp_id"], f"{name}  (AUC @ slippery, freeze={w['freeze_first']})")
plt.tight_layout(); plt.show()

## 6. Write Phase 2 results for Phase 3 to consume

In [ ]:
result = {
    "N":                  N,
    "X":                  X,
    "hp_seeds":           list(HP_SEEDS),
    "baseline_winners":   baseline_winners,
    "ft_winners":         ft_winners,
    "pretrained_checkpoints": {
        str(s): str(_pretrained_ckpt_path(s))
        for s in PRETRAIN_SEEDS if _pretrained_ckpt_path(s).exists()
    },
}

PHASE2_RESULTS.parent.mkdir(parents=True, exist_ok=True)
PHASE2_RESULTS.write_text(json.dumps(result, indent=2))
print(f"wrote {PHASE2_RESULTS}\n")

# Summary print: just the picks, not the full all_configs_auc dump.
summary = {
    "baseline_winners": {n: {k: v for k, v in w.items() if k != "all_configs_auc"}
                         for n, w in baseline_winners.items()},
    "ft_winners":       {n: {k: v for k, v in w.items() if k != "all_configs_auc"}
                         for n, w in ft_winners.items()},
    "pretrained_checkpoints": result["pretrained_checkpoints"],
}
print(json.dumps(summary, indent=2))

## 7. Phase 2 summary

If you got here cleanly:

- Winning HP per condition is locked in `runs/_phase2_results.json`.
- 3 pretrained checkpoints (one per HP_SEED) sit in `checkpoints/`.
- All eval logs are under `runs/phase2_search/` and `runs/phase2_finetune_search/` for reruns or re-scoring.

**For Phase 3** (per EXPERIMENT_PLAN §3):
1. Re-pretrain on normal with 5 fresh seeds using the winning B-normal HPs (§3.1).
2. For each of the 7 conditions, train 5 fresh seeds with the winning HPs at the appropriate budget (§3.2). Fine-tune conditions load the seed-matched fresh pretrained checkpoints.
3. Final eval per checkpoint uses `n_episodes=10` per plan §6 (we used 1 here because the env is deterministic and search-AUC is unaffected; bump for the final reported numbers to satisfy the spec).

## 8. Demo — watch the trained models drive

Two qualitative greedy rollouts on `straight_turn.txt`:

1. **Pretrained-on-normal** (`checkpoints/pretrained_normal_seed_{HP_SEEDS[0]}.pt`) at normal friction — the warm-start every fine-tune condition loads from.
2. **Best fine-tuned E1** (direct, full fine-tune) at slippery friction — the transfer-target setting.

Both saved as GIFs under `runs/_demos/` and displayed inline. Purely qualitative — the AUC numbers above are the source of truth for HP picking.

In [ ]:
from PIL import Image
from IPython.display import Image as IPyImage, display, Markdown

from experiment.env_utils import build_env
from experiment.dqn import DQNAgent
from experiment.run_io import load_checkpoint_into

def _record_episode(ckpt_path, map_path, friction, *, seed=1000, max_steps=2000, frame_stride=2):
    """Greedy rollout of `ckpt_path` on (map, friction). Returns (frames, total_return, last_info)."""
    env = build_env(map_path, friction=friction, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    agent = DQNAgent(
        obs_dim   = int(np.prod(env.observation_space.shape)),
        n_actions = env.action_space.n,
        hidden_sizes = [128, 128],   # plan §5: architecture fixed across all conditions
        device    = DEVICE,
    )
    load_checkpoint_into(ckpt_path, agent, load_target=False)

    frames = [env.render()]
    total, info = 0.0, {}
    for step in range(max_steps):
        a = agent.select_action(obs, greedy=True)
        obs, r, term, trunc, info = env.step(a)
        total += float(r)
        if step % frame_stride == 0:
            frames.append(env.render())
        if term or trunc:
            break
    env.close()
    return frames, total, info

def _save_gif(frames, path, fps=30):
    imgs = [Image.fromarray(f) for f in frames]
    imgs[0].save(path, save_all=True, append_images=imgs[1:],
                 duration=int(1000 / fps), loop=0, optimize=False)
    return path

DEMO_MAP = "maps/straight_turn.txt"
DEMO_DIR = pathlib.Path(SAVE_DIR) / "_demos"
DEMO_DIR.mkdir(parents=True, exist_ok=True)
demo_seed = HP_SEEDS[0]

# ---- demo 1: pretrained-on-normal warm-start ---------------------------
pretrained_ckpt = _pretrained_ckpt_path(demo_seed)
if pretrained_ckpt.exists():
    frames, ret, info = _record_episode(pretrained_ckpt, DEMO_MAP, FRICTION_NORMAL)
    gif = DEMO_DIR / f"phase2_pretrained_seed{demo_seed}.gif"
    _save_gif(frames, gif)
    display(Markdown(f"**Pretrained-on-normal** (seed={demo_seed}) on `{DEMO_MAP}` at f={FRICTION_NORMAL} — "
                     f"return={ret:+.1f}, finish={info.get('finish', False)}, "
                     f"crash={info.get('wall_hit', False)}, "
                     f"progress={info.get('progress_pct', 0):.1%}"))
    display(IPyImage(filename=str(gif)))
else:
    print(f"  [skip] no pretrained checkpoint at {pretrained_ckpt}")

# ---- demo 2: winning E1 fine-tuned (slippery target) -------------------
e1 = ft_winners.get("E1") if "ft_winners" in dir() else None
if e1 is not None:
    e1_run_dir = pathlib.Path(SAVE_DIR) / FT_SEARCH_SUBDIR / "E1" / e1["hp_id"] / f"seed_{demo_seed}"
    e1_ckpt = e1_run_dir / "final_checkpoint.pt"
    if e1_ckpt.exists():
        frames, ret, info = _record_episode(e1_ckpt, DEMO_MAP, FRICTION_SLIPPERY)
        gif = DEMO_DIR / f"phase2_E1_{e1['hp_id']}_seed{demo_seed}.gif"
        _save_gif(frames, gif)
        display(Markdown(f"**E1 fine-tuned** (`{e1['hp_id']}`, seed={demo_seed}) on `{DEMO_MAP}` at f={FRICTION_SLIPPERY} — "
                         f"return={ret:+.1f}, finish={info.get('finish', False)}, "
                         f"crash={info.get('wall_hit', False)}, "
                         f"progress={info.get('progress_pct', 0):.1%}"))
        display(IPyImage(filename=str(gif)))
    else:
        print(f"  [skip] no E1 final checkpoint at {e1_ckpt}")
else:
    print("  [skip] ft_winners not in scope — run section 4 first.")